### Nexora Care-Flow: Patient Volume Forecasting & Operational Staffing
#### COMPANY OVERVIEW
Nexora Health is a regional healthcare provider operating four outpatient clinics (*Nexora Care - Lakeside*, *Nexora Care - Riverside*, *Nexora Care - Downtown Express*, and *Nexora Care - Hillcrest*). The network offers a mix of primary care, multi-specialty consultations, and telehealth services. The organization's mission is to deliver high-quality, compassionate care while maintaining smooth clinic operations and minimal patient wait times.

#### PROJECT OVERVIEW
Nexora Care-Flow is an end-to-end data science initiative designed to assist clinic managers in optimizing physician and nursing shift schedules. By analyzing two years of historical appointment records (129,353 patient logs across 2024 and 2025), the project develops a predictive time-series forecasting pipeline that projects weekly patient visit volumes for each clinic. This provides clinic leadership with the advance visibility required to transition from reactive scheduling to proactive, data-driven workforce planning.

#### BUSINESS PROBLEM
Clinic directors face significant operational challenges due to unpredictable fluctuations in patient demand. High-volume periods—such as weekly appointment rushes or winter flu season spikes—frequently overload clinics, causing long waiting room delays, patient dissatisfaction, and clinician burnout. Conversely, holiday periods like Thanksgiving and Christmas experience sharp volume dips, leading to overstaffing, idle clinical hours, and unnecessary payroll expenses. Without reliable demand forecasts, management cannot align staff rotas with actual patient arrival patterns.

#### BUSINESS IMPACT
Implementing an accurate weekly patient forecasting model delivers key operational and financial advantages:
- Enhanced Patient Experience: Matching clinician coverage to patient volume reduces waiting room congestion and shortens wait times.
- Mitigated Staff Burnout: Balanced shift rotas during peak demand periods prevent doctor and nurse fatigue.
- Payroll & Cost Control: Eliminating overstaffing during holiday lulls prevents wasted labor expenditure.
- Proactive Operational Planning: Provides clinic managers with a 1-to-2 week advance notice window to finalize staffing schedules prior to appointment dates.

#### PROBLEM OBJECTIVE
The primary objective of the Nexora Care-Flow project is to build an operational, data-driven time-series forecasting system that accurately predicts weekly patient appointment volume across all four outpatient clinics.
Specifically, the project aims to:
- Clean & Prepare Data: Standardize raw appointment records into clean daily and weekly time-series datasets.
- Analyze Operational Drivers: Identify key demand patterns, including mid-week rushes, flu season peaks, and holiday volume dips.
- Develop Forecasting Models: Build and evaluate machine learning models to accurately predict weekly clinic appointment volume.
- Deliver Actionable Staffing Guidance: Translate predictions into concrete doctor and nurse shift recommendations and establish a long-term model maintenance framework.

In [257]:
# Load Python Libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

Load Raw Appointment Dataset

In [258]:
# Load Raw Dataset
df = pd.read_csv(r"C:\Users\earlc\Amdari\Week 2\nexora_care_flow\data\raw\AppointmentRecords.csv")
print(f"Loaded Raw Appointments Dataset into 'df': {len(df):,} rows")

Loaded Raw Appointments Dataset into 'df': 129,353 rows


In [259]:
# Inspect First 5 Rows
df.head()

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag
0,529385,2,Nexora Care - Lakeside,Multi-Specialty,103642,202,2024-01-01 08:00,New Patient,Completed,Online,2,True,New Year's Day,Flu Season,False
1,500000,1,Nexora Care - Riverside,Primary Care,101551,101,2024-01-01 08:30,New Patient,No-Show,Online,18,True,New Year's Day,Flu Season,False
2,500011,1,Nexora Care - Riverside,Primary Care,100260,104,2024-01-01 08:30,Follow-Up,Completed,Referral,4,True,New Year's Day,Flu Season,False
3,575066,3,Nexora Care - Downtown Express,Telehealth-Forward,104923,301,2024-01-01 09:45,Urgent,Completed,Online,0,True,New Year's Day,Flu Season,False
4,500008,1,Nexora Care - Riverside,Primary Care,101124,104,2024-01-01 09:45,Telehealth,Completed,Phone,2,True,New Year's Day,Flu Season,False


In [260]:
# Inspect Last 5 Rows
df.tail()

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag
129348,629350,4,Nexora Care - Hillcrest,Primary Care,105879,405,2025-12-31 16:30,Follow-Up,Completed,Online,12,True,New Year's Eve,Flu Season,False
129349,529373,1,Nexora Care - Riverside,Primary Care,101403,102,2025-12-31 16:30,Follow-Up,Completed,Online,7,True,New Year's Eve,Flu Season,True
129350,529376,1,Nexora Care - Riverside,Primary Care,101248,106,2025-12-31 16:30,Urgent,Completed,Online,2,True,New Year's Eve,Flu Season,False
129351,629337,4,Nexora Care - Hillcrest,Primary Care,106370,406,2025-12-31 16:45,Urgent,Completed,Online,1,True,New Year's Eve,Flu Season,False
129352,575056,2,Nexora Care - Lakeside,Multi-Specialty,102543,206,2025-12-31 17:45,Telehealth,Cancelled,Phone,10,True,New Year's Eve,Flu Season,True


In [261]:
# Dataset Shape
df.shape
print("Dataset Dimensions (Rows, Columns):", df.shape)

Dataset Dimensions (Rows, Columns): (129353, 15)


In [262]:
# Data Types & Column Summary
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   AppointmentID         129353 non-null  int64
 1   ClinicID              129353 non-null  int64
 2   ClinicName            129353 non-null  str  
 3   ClinicType            129353 non-null  str  
 4   PatientID             129353 non-null  int64
 5   ProviderID            129353 non-null  int64
 6   AppointmentDateTime   129353 non-null  str  
 7   AppointmentType       129353 non-null  str  
 8   Status                129353 non-null  str  
 9   BookingChannel        129353 non-null  str  
 10  BookingLeadTimeDays   129353 non-null  int64
 11  IsHoliday             129353 non-null  bool 
 12  HolidayName           939 non-null     str  
 13  SeasonFlag            129353 non-null  str  
 14  ChronicConditionFlag  129353 non-null  bool 
dtypes: bool(2), int64(5), str(8)
memory usage: 13

In [263]:
# Audit Summary
total_cols = df.shape[1]
total_rows = len(df)
dtypes_str = ", ".join([f"{count} {dtype}" for dtype, count in df.dtypes.value_counts().items()])
complete_cols_count = (df.notnull().sum() == total_rows).sum()
missing_cols = [
    f"'{col}' has {df[col].notnull().sum():,} non-null entries ({df[col].isnull().sum():,} missing values)"
    for col in df.columns if df[col].isnull().sum() > 0
]
missing_str = "; ".join(missing_cols) if missing_cols else "None (0 missing values across all columns)"
print(
    f"\n Audit Complete:\n"
    f"   - Total Columns: {total_cols}\n"
    f"   - Data Types: {dtypes_str}\n"
    f"   - Non-Null Count: {complete_cols_count} columns have full {total_rows:,} non-null entries.\n"
    f"   - Missing Data: {missing_str}"
)


 Audit Complete:
   - Total Columns: 15
   - Data Types: 8 str, 5 int64, 2 bool
   - Non-Null Count: 14 columns have full 129,353 non-null entries.
   - Missing Data: 'HolidayName' has 939 non-null entries (128,414 missing values)


In [264]:
# Audit Numerical Summary Statistics
df.describe()

,AppointmentID,ClinicID,PatientID,ProviderID,BookingLeadTimeDays
count,129353.000000,129353.000000,129353.000000,129353.000000,129353.000000
mean,564676.000000,2.417864,103553.485385,245.290005,8.704267
std,37341.139023,1.071525,2020.941152,107.165883,7.641921
min,500000.000000,1.000000,100000.000000,101.000000,0.000000
25%,532338.000000,2.000000,101812.000000,201.000000,3.000000
50%,564676.000000,2.000000,103547.000000,205.000000,7.000000
75%,597014.000000,3.000000,105178.000000,306.000000,12.000000
max,629352.000000,4.000000,107279.000000,406.000000,60.000000


In [265]:
# Summary
num_df = df.select_dtypes(include='number')
num_cols_count = num_df.shape[1]
num_col_names = ", ".join(num_df.columns)

# Calculate statistics for key numerical metric (BookingLeadTimeDays)
min_lead = df['BookingLeadTimeDays'].min()
mean_lead = df['BookingLeadTimeDays'].mean()
median_lead = df['BookingLeadTimeDays'].median()
max_lead = df['BookingLeadTimeDays'].max()

print(
    f"   Audit Complete:\n"
    f"   - Numerical Columns Analyzed ({num_cols_count}): {num_col_names}\n"
    f"   - Booking Lead Time Range: {min_lead} to {max_lead} days\n"
    f"   - Booking Lead Time Averages: Mean = {mean_lead:.2f} days | Median = {median_lead:.0f} days"
)


   Audit Complete:
   - Numerical Columns Analyzed (5): AppointmentID, ClinicID, PatientID, ProviderID, BookingLeadTimeDays
   - Booking Lead Time Range: 0 to 60 days
   - Booking Lead Time Averages: Mean = 8.70 days | Median = 7 days


In [266]:
# Unique Values Count
df.nunique()

AppointmentID           129353
ClinicID                     4
ClinicName                   4
ClinicType                   3
PatientID                 7109
ProviderID                  24
AppointmentDateTime      30287
AppointmentType              4
Status                       4
BookingChannel               4
BookingLeadTimeDays         61
IsHoliday                    2
HolidayName                 13
SeasonFlag                   3
ChronicConditionFlag         2
dtype: int64

In [267]:
# Summary
uniques = df.nunique()
total_rows = len(df)
pk_unique = "100% Unique Key" if uniques['AppointmentID'] == total_rows else "Duplicates Present"
low_card_cols = [col for col in df.columns if uniques[col] <= 5]

print(
    f" Audit Complete:\n"
    f"   - Unique Primary Key Check: {uniques['AppointmentID']:,} AppointmentIDs out of {total_rows:,} rows ({pk_unique})\n"
    f"   - Operational Entity Scale: {uniques['PatientID']:,} Unique Patients | {uniques['ProviderID']} Providers | {uniques['ClinicID']} Clinics\n"
    f"   - Low-Cardinality Categorical Breakdown:"
)

for col in low_card_cols:
    vals = df[col].dropna().unique().tolist()  # .tolist() converts np types to native Python int/bool/str
    print(f"     * {col}: {vals}")


 Audit Complete:
   - Unique Primary Key Check: 129,353 AppointmentIDs out of 129,353 rows (100% Unique Key)
   - Operational Entity Scale: 7,109 Unique Patients | 24 Providers | 4 Clinics
   - Low-Cardinality Categorical Breakdown:
     * ClinicID: [2, 1, 3, 4]
     * ClinicName: ['Nexora Care - Lakeside', 'Nexora Care - Riverside', 'Nexora Care - Downtown Express', 'Nexora Care - Hillcrest']
     * ClinicType: ['Multi-Specialty', 'Primary Care', 'Telehealth-Forward']
     * AppointmentType: ['New Patient', 'Follow-Up', 'Urgent', 'Telehealth']
     * Status: ['Completed', 'No-Show', 'Cancelled', 'Rescheduled']
     * BookingChannel: ['Online', 'Referral', 'Phone', 'Walk-In']
     * IsHoliday: [True, False]
     * SeasonFlag: ['Flu Season', 'Allergy Season', 'Standard']
     * ChronicConditionFlag: [False, True]


### Data Cleaning
#### Handling Missing Data

In [268]:
# Missing Values Audit
df.isnull().sum()

AppointmentID                0
ClinicID                     0
ClinicName                   0
ClinicType                   0
PatientID                    0
ProviderID                   0
AppointmentDateTime          0
AppointmentType              0
Status                       0
BookingChannel               0
BookingLeadTimeDays          0
IsHoliday                    0
HolidayName             128414
SeasonFlag                   0
ChronicConditionFlag         0
dtype: int64

In [269]:
# Summary
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
total_rows = len(df)
total_cols = df.shape[1]

complete_cols_count = (null_counts == 0).sum()
missing_series = null_counts[null_counts > 0]

missing_details = [
    f"'{col}' ({count:,} missing / {count / total_rows:.2%})"
    for col, count in missing_series.items()
]
missing_str = ", ".join(missing_details) if missing_details else "None (0 missing values across all columns)"

print(
    f"Audit Complete:\n"
    f"   - Total Missing Values: {total_nulls:,} missing cells across entire dataset\n"
    f"   - Column Completeness: {complete_cols_count} of {total_cols} columns have 0 missing values (100% complete)\n"
    f"   - Columns Requiring Imputation: {missing_str}"
)


Audit Complete:
   - Total Missing Values: 128,414 missing cells across entire dataset
   - Column Completeness: 14 of 15 columns have 0 missing values (100% complete)
   - Columns Requiring Imputation: 'HolidayName' (128,414 missing / 99.27%)


In [270]:
# Impute Missing HolidayName Values
# Missing values in `HolidayName` represent standard non-holiday operating days. Imputing `'Non-Holiday'` (rather than `'None'`, which Pandas `read_csv` parses back as null) ensures the exported CSV is 100% clean with 0 missing values across all records.
df['HolidayName'] = df['HolidayName'].fillna('Non-Holiday')

# Remaining Missing Values after Holiday Imputation
df.isnull().sum()

AppointmentID           0
ClinicID                0
ClinicName              0
ClinicType              0
PatientID               0
ProviderID              0
AppointmentDateTime     0
AppointmentType         0
Status                  0
BookingChannel          0
BookingLeadTimeDays     0
IsHoliday               0
HolidayName             0
SeasonFlag              0
ChronicConditionFlag    0
dtype: int64

In [271]:
# Summary
imputed_count = (df['HolidayName'] == 'Non-Holiday').sum()
remaining_nulls = df.isnull().sum().sum()
total_rows = len(df)
complete_cols_count = (df.isnull().sum() == 0).sum()
total_cols = df.shape[1]

print(
    f"Action Complete:\n"
    f"   - Imputation Value Applied: 'Non-Holiday' assigned to {imputed_count:,} records ({imputed_count / total_rows:.2%})\n"
    f"   - Dataset Hygiene: {complete_cols_count} of {total_cols} columns now have 0 missing values (100% complete)\n"
    f"   - Remaining Nulls Audit: {remaining_nulls} missing values across all records"
)


Action Complete:
   - Imputation Value Applied: 'Non-Holiday' assigned to 128,414 records (99.27%)
   - Dataset Hygiene: 15 of 15 columns now have 0 missing values (100% complete)
   - Remaining Nulls Audit: 0 missing values across all records


#### Checking for duplicates

In [272]:
# Check for Duplicates
df.duplicated().sum()

np.int64(0)

In [273]:
# Summary
initial_rows = len(df)
df = df.drop_duplicates(subset=['AppointmentID']).reset_index(drop=True)
final_rows = len(df)
duplicates_removed = initial_rows - final_rows
dedup_status = "100% Unique (Zero Duplicates)" if duplicates_removed == 0 else f"{duplicates_removed:,} Duplicates Removed"

print(
    f"Action Complete:\n"
    f"   - Initial Record Count: {initial_rows:,} rows audited\n"
    f"   - Deduplication Status: {duplicates_removed} duplicate AppointmentIDs found ({dedup_status})\n"
    f"   - Final Clean Row Count: {final_rows:,} records ready for downstream modeling"
)


Action Complete:
   - Initial Record Count: 129,353 rows audited
   - Deduplication Status: 0 duplicate AppointmentIDs found (100% Unique (Zero Duplicates))
   - Final Clean Row Count: 129,353 records ready for downstream modeling


#### Inspect & Convert Datetime Column

In [274]:
# Check Data Type BEFORE -> Convert Datetime -> Verify Data Type AFTER

# Step 1: Capture data type BEFORE conversion
dtype_before = df['AppointmentDateTime'].dtype

# Step 2: Convert string timestamp to pandas datetime object
df['AppointmentDateTime'] = pd.to_datetime(df['AppointmentDateTime'])

# Step 3: Capture data type AFTER conversion & extract date range
dtype_after = df['AppointmentDateTime'].dtype
min_date = df['AppointmentDateTime'].min()
max_date = df['AppointmentDateTime'].max()
total_days = (max_date - min_date).days

# Audit Summary
print(
    f" Action Complete:\n"
    f"   - Target Column: 'AppointmentDateTime'\n"
    f"   - Type Conversion: Converted from '{dtype_before}' ➔ '{dtype_after}'\n"
    f"   - Verified Date Range: {min_date:%Y-%m-%d} to {max_date:%Y-%m-%d} ({total_days:,} days span)"
)

df.head()


 Action Complete:
   - Target Column: 'AppointmentDateTime'
   - Type Conversion: Converted from 'str' ➔ 'datetime64[us]'
   - Verified Date Range: 2024-01-01 to 2025-12-31 (730 days span)


,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag
0,529385,2,Nexora Care - Lakeside,Multi-Specialty,103642,202,2024-01-01 08:00:00,New Patient,Completed,Online,2,True,New Year's Day,Flu Season,False
1,500000,1,Nexora Care - Riverside,Primary Care,101551,101,2024-01-01 08:30:00,New Patient,No-Show,Online,18,True,New Year's Day,Flu Season,False
2,500011,1,Nexora Care - Riverside,Primary Care,100260,104,2024-01-01 08:30:00,Follow-Up,Completed,Referral,4,True,New Year's Day,Flu Season,False
3,575066,3,Nexora Care - Downtown Express,Telehealth-Forward,104923,301,2024-01-01 09:45:00,Urgent,Completed,Online,0,True,New Year's Day,Flu Season,False
4,500008,1,Nexora Care - Riverside,Primary Care,101124,104,2024-01-01 09:45:00,Telehealth,Completed,Phone,2,True,New Year's Day,Flu Season,False


In [275]:
output_path = r"C:\Users\earlc\Amdari\Week 2\nexora_care_flow\data\processed\cleaned_appointments.csv"

# Export Cleaned Record-Level Dataset to Processed Folder
df.to_csv(output_path, index=False)

# Audit Summary
export_rows = len(df)
export_cols = df.shape[1]
total_nulls = df.isnull().sum().sum()

print(
    f" Action Complete:\n"
    f"   - Export Destination: '{output_path}'\n"
    f"   - Quality Guarantee: 100% clean with {total_nulls} missing values across all records\n"
)

df.info()


 Action Complete:
   - Export Destination: 'C:\Users\earlc\Amdari\Week 2\nexora_care_flow\data\processed\cleaned_appointments.csv'
   - Quality Guarantee: 100% clean with 0 missing values across all records

<class 'pandas.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   AppointmentID         129353 non-null  int64         
 1   ClinicID              129353 non-null  int64         
 2   ClinicName            129353 non-null  str           
 3   ClinicType            129353 non-null  str           
 4   PatientID             129353 non-null  int64         
 5   ProviderID            129353 non-null  int64         
 6   AppointmentDateTime   129353 non-null  datetime64[us]
 7   AppointmentType       129353 non-null  str           
 8   Status                129353 non-null  str           
 9   BookingChannel        129353 non-null 